<a href="https://colab.research.google.com/github/balasri03/Mini_project/blob/main/review_pstance.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import re
import torch
from torch.nn.utils.rnn import pad_sequence
from collections import Counter
import pandas as pd
import time
from collections import Counter
import numpy as np

In [ ]:
train_df = pd.read_csv("trainmerged (4).csv")
val_df = pd.read_csv("/content/valmerged (3).csv")
test_df = pd.read_csv("/content/testmerged (3).csv")

In [ ]:
train_df.shape

In [ ]:
print(train_df.columns)

In [ ]:
print(train_df["Stance"].value_counts())

In [ ]:
print(len(train_df))
print(len(val_df))
print(len(test_df))

In [ ]:
import re

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+", "", text)          # remove URLs
    text = re.sub(r"@\w+", "", text)                    # remove mentions
    text = re.sub(r"#", "", text)                       # remove hashtags
    text = re.sub(r"[^a-z\s]", "", text)                # keep only letters
    text = re.sub(r"\s+", " ", text).strip()            # remove extra spaces
    return text


train_df["clean_text"] = (train_df["Tweet"] + " " + train_df["Target"]).apply(clean_text)

val_df["clean_text"] = (val_df["Tweet"] + " " + val_df["Target"]).apply(clean_text)

test_df["clean_text"] = (test_df["Tweet"] + " " + test_df["Target"]).apply(clean_text)

In [ ]:
from collections import Counter

def build_vocab(texts, min_freq=5):
    counter = Counter()

    for text in texts:
        words = text.split()
        counter.update(words)

    vocab = {"<pad>": 0, "<unk>": 1}

    for word, freq in counter.items():
        if freq >= min_freq:
            vocab[word] = len(vocab)

    return vocab


vocab = build_vocab(train_df["clean_text"])

In [ ]:
def text_to_seq(text, vocab):
    words = text.split()

    return [
        vocab.get(word, vocab["<unk>"])
        for word in words
    ]

In [ ]:
train_df["input_text"] = train_df["clean_text"] + " " + train_df["Target"]
val_df["input_text"]   = val_df["clean_text"]   + " " + val_df["Target"]
test_df["input_text"]  = test_df["clean_text"]  + " " + test_df["Target"]

vocab = build_vocab(train_df["input_text"].tolist())

print(f"Vocab size: {len(vocab)}")

In [ ]:
len(train_df['input_text'])

In [ ]:
label_col = "Stance"

unique_labels = sorted(train_df[label_col].unique())

label2id = {label: idx for idx, label in enumerate(unique_labels)}

id2label = {v: k for k, v in label2id.items()}

print("Label mapping:", label2id)

In [ ]:
train = train_df["input_text"]
len(train)

In [ ]:
test = test_df["input_text"]
len(test)

In [ ]:
val = val_df["input_text"]
len(val)

LSTM BASELINE

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense
from tensorflow.keras.layers import Input, Embedding
from tensorflow.keras.layers import Dropout
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.layers import SpatialDropout1D

In [ ]:
X_train = [text_to_seq(text, vocab) for text in train_df["input_text"]]
X_val   = [text_to_seq(text, vocab) for text in val_df["input_text"]]
X_test  = [text_to_seq(text, vocab) for text in test_df["input_text"]]

In [ ]:
max_len = 50

X_train_pad = pad_sequences(X_train, maxlen=max_len, padding="post")
X_val_pad   = pad_sequences(X_val, maxlen=max_len, padding="post")
X_test_pad  = pad_sequences(X_test, maxlen=max_len, padding="post")

In [ ]:
y_train = train_df["Stance"].map(label2id).values
y_val   = val_df["Stance"].map(label2id).values
y_test  = test_df["Stance"].map(label2id).values

In [ ]:
vocab_size = len(vocab)
embedding_dim = 50

model = Sequential()

model.add(Input(shape=(max_len,)))

model.add(Embedding(
    input_dim=vocab_size,
    output_dim=embedding_dim
))

model.add(SpatialDropout1D(0.2))

model.add(LSTM(32, dropout=0.2, recurrent_dropout=0.2))



model.add(Dense(len(label2id), activation="softmax"))

model.compile(

    loss="sparse_categorical_crossentropy",

    optimizer="adam",

    metrics=["accuracy"]

)

model.summary()

In [ ]:
early_stop = EarlyStopping(

    monitor="val_loss",

    patience=3,

    restore_best_weights=True

)

history = model.fit(

    X_train_pad,
    y_train,

    validation_data=(X_val_pad, y_val),

    epochs=8,

    batch_size=32,

    callbacks=[early_stop]

)

In [ ]:
from sklearn.metrics import roc_curve, auc
import matplotlib.pyplot as plt

# Predict probabilities
y_pred_prob = model.predict(X_test_pad)

# If output is softmax (2 classes)
y_pred_prob = y_pred_prob[:,1]

# ROC calculations
fpr, tpr, thresholds = roc_curve(y_test, y_pred_prob)
roc_auc = auc(fpr, tpr)

print("LSTM Baseline AUC Score:", roc_auc)

# Plot ROC Curve
plt.figure(figsize=(6,5))
plt.plot(fpr, tpr, label="LSTM Baseline (AUC = %0.3f)" % roc_auc)
plt.plot([0,1], [0,1], linestyle="--")

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve - LSTM Baseline Model")
plt.legend()
plt.show()

In [ ]:
loss, acc = model.evaluate(X_test_pad, y_test)

print("Baseline Test Accuracy:", acc)

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
import numpy as np

# Predict probabilities
y_pred_probs = model.predict(X_test_pad)

# Convert to class labels
y_pred = np.argmax(y_pred_probs, axis=1)

In [ ]:
cm = confusion_matrix(y_test, y_pred)

print("Confusion Matrix:\n")
print(cm)

one hot

In [ ]:
# STEP 1: text → sequence

X_train = [text_to_seq(text, vocab) for text in train_df["input_text"]]

X_val   = [text_to_seq(text, vocab) for text in val_df["input_text"]]

X_test  = [text_to_seq(text, vocab) for text in test_df["input_text"]]

In [ ]:
# STEP 2: limit vocab

max_words = 2000

X_train_oh = [[min(i, max_words-1) for i in seq] for seq in X_train]

X_val_oh   = [[min(i, max_words-1) for i in seq] for seq in X_val]

X_test_oh  = [[min(i, max_words-1) for i in seq] for seq in X_test]

In [ ]:
# STEP 3: padding

from tensorflow.keras.preprocessing.sequence import pad_sequences

max_len_oh = 40

X_train_pad_oh = pad_sequences(X_train_oh, maxlen=max_len_oh, padding="post")

X_val_pad_oh   = pad_sequences(X_val_oh, maxlen=max_len_oh, padding="post")

X_test_pad_oh  = pad_sequences(X_test_oh, maxlen=max_len_oh, padding="post")

In [ ]:
# STEP 4

from tensorflow.keras.layers import Lambda
import tensorflow.keras.backend as K

In [ ]:
# STEP 5

vocab_size_oh = max_words

def one_hot_layer(x):

    return K.one_hot(x, vocab_size_oh)

In [ ]:
# STEP 6

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Input, Dropout

onehot_model = Sequential()

onehot_model.add(Input(shape=(max_len_oh,)))

onehot_model.add(Lambda(one_hot_layer))

onehot_model.add(LSTM(32, dropout=0.3, recurrent_dropout=0.3))

onehot_model.add(Dense(32, activation="relu"))

onehot_model.add(Dropout(0.3))

onehot_model.add(Dense(1, activation="sigmoid"))

onehot_model.compile(

    loss="binary_crossentropy",

    optimizer="adam",

    metrics=["accuracy"]

)

onehot_model.summary()

In [ ]:
# STEP 7

history_onehot = onehot_model.fit(

    X_train_pad_oh,
    y_train,

    validation_data=(X_val_pad_oh, y_val),

    epochs=10,

    batch_size=64

)

In [ ]:
loss, acc = onehot_model.evaluate(X_test_pad_oh, y_test)

print("One Hot Test Accuracy:", acc)

In [ ]:
from sklearn.metrics import confusion_matrix

y_pred = (onehot_model.predict(X_test_pad_oh) > 0.5).astype("int32")

print(confusion_matrix(y_test, y_pred))

In [ ]:
from sklearn.metrics import roc_curve, auc
import matplotlib.pyplot as plt

# Predict probabilities
y_pred_prob = onehot_model.predict(X_test_pad)

# Convert to 1D
y_pred_prob = y_pred_prob.ravel()

# ROC calculation
fpr, tpr, thresholds = roc_curve(y_test, y_pred_prob)
roc_auc = auc(fpr, tpr)

print("One-Hot AUC Score:", roc_auc)

# Plot ROC
plt.figure(figsize=(6,5))
plt.plot(fpr, tpr, label="One-Hot (AUC = %0.3f)" % roc_auc)
plt.plot([0,1], [0,1], linestyle="--")

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve - One-Hot Model")
plt.legend()
plt.show()

GLOVE EMBEDDINGS

In [ ]:
!pip install gensim

In [ ]:
import gensim.downloader as api
import numpy as np

In [ ]:
glove = api.load("glove-twitter-100")

embedding_dim = 100

print("GloVe loaded")

In [ ]:
vocab_size = len(vocab)

embedding_matrix = np.zeros((vocab_size, embedding_dim))

for word, index in vocab.items():

    if word in glove:

        embedding_matrix[index] = glove[word]

print("Embedding matrix ready")

In [ ]:
X_train_glove = X_train_pad
X_val_glove   = X_val_pad
X_test_glove  = X_test_pad

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Input, Dropout

glove_model = Sequential()

glove_model.add(Input(shape=(max_len,)))

glove_model.add(Embedding(

    input_dim=vocab_size,

    output_dim=embedding_dim,

    weights=[embedding_matrix],

    trainable=True

))

glove_model.add(LSTM(80, dropout=0.2, recurrent_dropout=0.2))

glove_model.add(Dense(32, activation="relu"))

glove_model.add(Dropout(0.2))

glove_model.add(Dense(1, activation="sigmoid"))

glove_model.compile(

    loss="binary_crossentropy",

    optimizer="adam",

    metrics=["accuracy"]

)

glove_model.summary()

In [ ]:
history_glove = glove_model.fit(

    X_train_glove,
    y_train,

    validation_data=(X_val_glove, y_val),

    epochs=4,

    batch_size=64

)

In [ ]:
from sklearn.metrics import roc_curve, auc
import matplotlib.pyplot as plt

# Predict probabilities
y_pred_prob = glove_model.predict(X_test_pad)

# Convert predictions to 1D
y_pred_prob = y_pred_prob.ravel()

# ROC calculation
fpr, tpr, thresholds = roc_curve(y_test, y_pred_prob)
roc_auc = auc(fpr, tpr)

print("GloVe Model AUC Score:", roc_auc)

# Plot ROC Curve
plt.figure(figsize=(6,5))
plt.plot(fpr, tpr, label="GloVe Model (AUC = %0.3f)" % roc_auc)
plt.plot([0,1], [0,1], linestyle="--")

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve - GloVe Model")
plt.legend()
plt.show()

In [ ]:
loss, acc = glove_model.evaluate(X_test_glove, y_test)

print("FINAL GLOVE TEST ACCURACY:", acc)

In [ ]:
from sklearn.metrics import confusion_matrix

y_pred = (glove_model.predict(X_test_glove) > 0.5).astype("int32")

print(confusion_matrix(y_test, y_pred))

ATTENTION LAYER

In [ ]:
import tensorflow as tf
from tensorflow.keras.layers import Layer
from tensorflow.keras.layers import Input, Embedding, LSTM, Dense, Dropout
from tensorflow.keras.models import Model

In [ ]:
class AttentionLayer(Layer):

    def __init__(self, **kwargs):
        super(AttentionLayer, self).__init__(**kwargs)

    def build(self, input_shape):

        self.W = self.add_weight(

            name="att_weight",

            shape=(input_shape[-1], 1),

            initializer="normal"

        )

        self.b = self.add_weight(

            name="att_bias",

            shape=(input_shape[1], 1),

            initializer="zeros"

        )

        super().build(input_shape)


    def call(self, x):

        e = tf.keras.backend.tanh(

            tf.keras.backend.dot(x, self.W) + self.b

        )

        a = tf.keras.backend.softmax(e, axis=1)

        output = x * a

        return tf.keras.backend.sum(output, axis=1)

In [ ]:
input_layer = Input(shape=(max_len,))

embedding = Embedding(

    input_dim=vocab_size,

    output_dim=embedding_dim,

    weights=[embedding_matrix],

    trainable=True

)(input_layer)


lstm = LSTM(

    64,

    return_sequences=True,

    dropout=0.2,

    recurrent_dropout=0.2

)(embedding)


attention = AttentionLayer()(lstm)


dense = Dense(32, activation="relu")(attention)

drop = Dropout(0.2)(dense)

output = Dense(1, activation="sigmoid")(drop)


attention_model = Model(inputs=input_layer, outputs=output)


attention_model.compile(

    loss="binary_crossentropy",

    optimizer="adam",

    metrics=["accuracy"]

)

attention_model.summary()

In [ ]:
history_att = attention_model.fit(

    X_train_glove,
    y_train,

    validation_data=(X_val_glove, y_val),

    epochs=5,

    batch_size=64

)

In [ ]:
loss, acc = attention_model.evaluate(X_test_glove, y_test)

print("FINAL ATTENTION TEST ACCURACY:", acc)

In [ ]:
from sklearn.metrics import confusion_matrix

y_pred = (attention_model.predict(X_test_glove) > 0.5).astype("int32")

print(confusion_matrix(y_test, y_pred))

In [ ]:
from sklearn.metrics import roc_curve, auc
import matplotlib.pyplot as plt

In [ ]:
y_pred_prob = attention_model.predict(X_test_glove)

In [ ]:
fpr, tpr, thresholds = roc_curve(y_test, y_pred_prob)

roc_auc = auc(fpr, tpr)

print("AUC Score:", roc_auc)

In [ ]:
plt.figure()

plt.plot(fpr, tpr, label="Attention + GloVe (AUC = %0.3f)" % roc_auc)

plt.plot([0,1], [0,1], linestyle="--")

plt.xlabel("False Positive Rate")

plt.ylabel("True Positive Rate")

plt.title("ROC Curve – Attention + GloVe")

plt.legend()

plt.show()

BASELINE GRU

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GRU, Dense, Input, Embedding, Dropout

In [ ]:
X_train_gru = X_train_pad
X_val_gru   = X_val_pad
X_test_gru  = X_test_pad

In [ ]:
# from tensorflow.keras.models import Sequential
# from tensorflow.keras.layers import GRU, Dense, Input, Embedding

# gru_model = Sequential()

# gru_model.add(Input(shape=(max_len,)))

# gru_model.add(Embedding(

#     input_dim=len(vocab),

#     output_dim=16   # reduced

# ))

# # gru_model.add(GRU(

# #     8   # reduced strongly

# # ))
# gru_model.add(GRU(8, dropout=0.3, recurrent_dropout=0.3))
# gru_model.add(Dense(1, activation="sigmoid"))

# gru_model.compile(

#     loss="binary_crossentropy",

#     optimizer="adam",

#     metrics=["accuracy"]

# )

# gru_model.summary()
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GRU, Dense, Input, Embedding

gru_model = Sequential()

gru_model.add(Input(shape=(max_len,)))

gru_model.add(Embedding(

    input_dim=len(vocab),

    output_dim=16   # reduced embedding

))

gru_model.add(GRU(

    8   # reduced GRU units

))

gru_model.add(Dense(1, activation="sigmoid"))

gru_model.compile(

    loss="binary_crossentropy",

    optimizer="adam",

    metrics=["accuracy"]

)

gru_model.summary()

In [ ]:
# history_gru = gru_model.fit(

#     X_train_pad,
#     y_train,

#     validation_data=(X_val_pad, y_val),

#     epochs=4,

#     batch_size=64

# )
history_gru = gru_model.fit(

    X_train_pad,
    y_train,

    validation_data=(X_val_pad, y_val),

    epochs=3,

    batch_size=64

)

In [ ]:
# Baseline GRU Test Accuracy

loss, acc = gru_model.evaluate(X_test_pad, y_test)

print("Baseline GRU Test Accuracy:", acc)

In [ ]:
# Predict class labels

y_pred_prob = gru_model.predict(X_test_pad)

y_pred = (y_pred_prob > 0.5).astype("int32")

In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_pred)

print("Confusion Matrix:\n")

print(cm)

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred))

In [ ]:
from sklearn.metrics import roc_curve, auc
import matplotlib.pyplot as plt

# Predict probabilities
y_pred_prob = gru_model.predict(X_test_pad)

# Convert to 1D
y_pred_prob = y_pred_prob.ravel()

# ROC calculation
fpr, tpr, thresholds = roc_curve(y_test, y_pred_prob)
roc_auc = auc(fpr, tpr)

print("GRU Baseline AUC Score:", roc_auc)

# Plot ROC Curve
plt.figure(figsize=(6,5))
plt.plot(fpr, tpr, label="GRU Baseline (AUC = %0.3f)" % roc_auc)
plt.plot([0,1], [0,1], linestyle="--")

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve - GRU Baseline")
plt.legend()
plt.show()

ONE HOT

In [ ]:
# ONE HOT BLOCK 1: Imports

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input, Lambda, Dropout, GRU
from tensorflow.keras.preprocessing.sequence import pad_sequences
import tensorflow.keras.backend as K
import numpy as np

In [ ]:
# ONE HOT BLOCK 2: Convert text to sequence

X_train = [text_to_seq(text, vocab) for text in train_df["input_text"]]

X_val   = [text_to_seq(text, vocab) for text in val_df["input_text"]]

X_test  = [text_to_seq(text, vocab) for text in test_df["input_text"]]

In [ ]:
# # ONE HOT BLOCK 3: Limit vocab

# max_words = 2000

# X_train_oh = [[min(i, max_words-1) for i in seq] for seq in X_train]

# X_val_oh   = [[min(i, max_words-1) for i in seq] for seq in X_val]

# X_test_oh  = [[min(i, max_words-1) for i in seq] for seq in X_test]
max_words = 5000

X_train_oh = [[min(i, max_words-1) for i in seq] for seq in X_train]
X_val_oh   = [[min(i, max_words-1) for i in seq] for seq in X_val]
X_test_oh  = [[min(i, max_words-1) for i in seq] for seq in X_test]

In [ ]:
# ONE HOT BLOCK 4: Padding

max_len_oh = 40

X_train_pad_oh = pad_sequences(X_train_oh, maxlen=max_len_oh, padding="post")

X_val_pad_oh   = pad_sequences(X_val_oh, maxlen=max_len_oh, padding="post")

X_test_pad_oh  = pad_sequences(X_test_oh, maxlen=max_len_oh, padding="post")

In [ ]:
# # ONE HOT BLOCK 5: One Hot Layer

# vocab_size_oh = max_words

# def one_hot_layer(x):

#     return K.one_hot(x, vocab_size_oh)

In [ ]:
# # ONE HOT BLOCK 6: Model

# onehot_model = Sequential()

# onehot_model.add(Input(shape=(max_len_oh,)))

# onehot_model.add(Lambda(one_hot_layer))

# onehot_model.add(GRU(64, dropout=0.3, recurrent_dropout=0.3))

# onehot_model.add(Dense(32, activation="relu"))

# onehot_model.add(Dropout(0.3))

# onehot_model.add(Dense(1, activation="sigmoid"))

# onehot_model.compile(

#     loss="binary_crossentropy",

#     optimizer="adam",

#     metrics=["accuracy"]

# )

# onehot_model.summary()
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input, Lambda, GRU, TimeDistributed
import tensorflow.keras.backend as K

vocab_size_oh = max_words

def one_hot_layer(x):
    return K.one_hot(x, vocab_size_oh)

onehot_model = Sequential()

onehot_model.add(Input(shape=(max_len_oh,)))

# One-Hot
onehot_model.add(Lambda(one_hot_layer))

# Trainable projection ← THIS improves accuracy
onehot_model.add(TimeDistributed(Dense(128, activation="relu")))

# GRU
onehot_model.add(GRU(128))

# Output
onehot_model.add(Dense(1, activation="sigmoid"))

onehot_model.compile(
    loss="binary_crossentropy",
    optimizer="adam",
    metrics=["accuracy"]
)

onehot_model.summary()

In [ ]:
# # ONE HOT BLOCK 7: Training

# history_onehot = onehot_model.fit(

#     X_train_pad_oh,
#     y_train,

#     validation_data=(X_val_pad_oh, y_val),

#     epochs=8,

#     batch_size=64

# )
history_onehot = onehot_model.fit(
    X_train_pad_oh,
    y_train,
    validation_data=(X_val_pad_oh, y_val),
    epochs=6,
    batch_size=64
)

In [ ]:
# ONE-HOT Test Accuracy

loss, acc = onehot_model.evaluate(X_test_pad_oh, y_test)

print("One-Hot Test Accuracy:", acc)

In [ ]:
# Predict probabilities

y_pred_prob = onehot_model.predict(X_test_pad_oh)

# Convert to class labels

y_pred = (y_pred_prob > 0.5).astype("int32")

In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_pred)

print("Confusion Matrix:\n")

print(cm)

In [ ]:
from sklearn.metrics import roc_curve, auc
import matplotlib.pyplot as plt

y_prob = onehot_model.predict(X_test_pad_oh)

fpr, tpr, _ = roc_curve(y_test, y_prob)

roc_auc = auc(fpr, tpr)

plt.plot(fpr, tpr, label="One-Hot AUC=%0.3f" % roc_auc)
plt.plot([0,1],[0,1],'--')
plt.legend()
plt.show()

GLOVE EMBEDDINGS

In [ ]:
!pip install gensim

import gensim.downloader as api
import numpy as np

In [ ]:
vocab_size = len(vocab)

embedding_matrix = np.zeros((vocab_size, embedding_dim))

for word, index in vocab.items():

    if word in glove:
        embedding_matrix[index] = glove[word]

print("Embedding matrix ready")

In [ ]:
X_train_glove = X_train_pad
X_val_glove   = X_val_pad
X_test_glove  = X_test_pad

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, GRU, Dense, Dropout, Input

glove_model = Sequential()

glove_model.add(Input(shape=(max_len,)))

glove_model.add(Embedding(

    input_dim=vocab_size,

    output_dim=embedding_dim,

    weights=[embedding_matrix],

    trainable=True   # IMPORTANT for higher accuracy

))

glove_model.add(GRU(128, dropout=0.2, recurrent_dropout=0.2))

glove_model.add(Dense(64, activation="relu"))

glove_model.add(Dropout(0.3))

glove_model.add(Dense(1, activation="sigmoid"))

glove_model.compile(

    loss="binary_crossentropy",

    optimizer="adam",

    metrics=["accuracy"]

)

glove_model.summary()

In [ ]:
history_glove = glove_model.fit(

    X_train_glove,
    y_train,

    validation_data=(X_val_glove, y_val),

    epochs=6,

    batch_size=64

)

In [ ]:
loss, acc = glove_model.evaluate(X_test_glove, y_test)

print("Final GloVe Test Accuracy:", acc)

In [ ]:
from sklearn.metrics import confusion_matrix

y_pred = (glove_model.predict(X_test_glove) > 0.5).astype("int32")

print(confusion_matrix(y_test, y_pred))

In [ ]:
from sklearn.metrics import roc_curve, auc
import matplotlib.pyplot as plt

In [ ]:
y_prob_glove = glove_model.predict(X_test_glove)

In [ ]:
fpr, tpr, thresholds = roc_curve(y_test, y_prob_glove)

roc_auc = auc(fpr, tpr)

print("GloVe AUC Score:", roc_auc)

In [ ]:
plt.figure()

plt.plot(fpr, tpr, label="GloVe GRU (AUC = %0.3f)" % roc_auc)

plt.plot([0,1], [0,1], linestyle="--")

plt.xlabel("False Positive Rate")

plt.ylabel("True Positive Rate")

plt.title("ROC Curve – GloVe Model")

plt.legend()

plt.show()

ATTENTION LAYER

In [ ]:
import tensorflow as tf
from tensorflow.keras.layers import Layer, Input, Embedding, GRU, Dense, Dropout
from tensorflow.keras.models import Model

In [ ]:
class AttentionLayer(Layer):
    def __init__(self):
        super(AttentionLayer, self).__init__()

    def build(self, input_shape):

        self.W = self.add_weight(
            name="att_weight",
            shape=(input_shape[-1], 1),
            initializer="normal"
        )

        self.b = self.add_weight(
            name="att_bias",
            shape=(input_shape[1], 1),
            initializer="zeros"
        )

        super(AttentionLayer, self).build(input_shape)

    def call(self, x):

        e = tf.keras.backend.tanh(
            tf.keras.backend.dot(x, self.W) + self.b
        )

        a = tf.keras.backend.softmax(e, axis=1)

        context = x * a
        context = tf.keras.backend.sum(context, axis=1)

        return context, a

In [ ]:
input_layer = Input(shape=(max_len,))

embedding = Embedding(

    input_dim=vocab_size,
    output_dim=embedding_dim,
    weights=[embedding_matrix],
    trainable=True

)(input_layer)


gru_out = GRU(

    128,
    return_sequences=True,
    dropout=0.2,
    recurrent_dropout=0.2

)(embedding)


attention_out, attention_weights = AttentionLayer()(gru_out)


dense = Dense(64, activation="relu")(attention_out)

drop = Dropout(0.3)(dense)

output = Dense(1, activation="sigmoid")(drop)


attention_model = Model(inputs=input_layer, outputs=output)


attention_model.compile(

    loss="binary_crossentropy",
    optimizer="adam",
    metrics=["accuracy"]

)

attention_model.summary()

In [ ]:
history_att = attention_model.fit(

    X_train_glove,
    y_train,

    validation_data=(X_val_glove, y_val),

    epochs=5,

    batch_size=64

)

In [ ]:
loss, acc = attention_model.evaluate(X_test_glove, y_test)

print("Final Attention Test Accuracy:", acc)

In [ ]:
attention_extractor = Model(
    inputs=attention_model.input,
    outputs=attention_weights
)

In [ ]:
attention_scores = attention_extractor.predict(X_test_glove[0:1])

In [ ]:
sequence = X_test_glove[0]

words = ["token_"+str(i) for i in range(len(sequence))]

attention_scores = attention_scores[0]

In [ ]:
from sklearn.metrics import confusion_matrix

y_pred = (attention_model.predict(X_test_glove) > 0.5).astype("int32")

print("Attention Confusion Matrix:")

print(confusion_matrix(y_test, y_pred))

In [ ]:
from sklearn.metrics import roc_curve, auc
import matplotlib.pyplot as plt

y_prob = attention_model.predict(X_test_glove)

fpr, tpr, _ = roc_curve(y_test, y_prob)

roc_auc = auc(fpr, tpr)

print("Attention AUC:", roc_auc)

plt.plot(fpr, tpr, label="Attention (AUC=%0.3f)" % roc_auc)
plt.plot([0,1],[0,1],'--')
plt.legend()
plt.title("ROC Curve - Attention Model")
plt.show()

In [ ]:
valid_len = (X_test_glove[0] != 0).sum()

attention_scores = attention_scores[:valid_len]

words = ["token_"+str(i) for i in range(valid_len)]

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(14,2))

sns.heatmap(
    attention_scores.T, # Changed from attention_scores[0].T to attention_scores.T
    cmap=sns.light_palette("green", as_cmap=True),
    xticklabels=words,
    cbar=True
)

plt.title("Attention Heatmap")
plt.xticks(rotation=90)
plt.show()

Mini-Lstm


Mini-Lstm+Glove


In [ ]:
vocab = build_vocab(train_df["input_text"])

In [ ]:
!wget http://nlp.stanford.edu/data/glove.6B.zip

In [ ]:
!unzip glove.6B.zip

In [ ]:
glove_path = "glove.6B.100d.txt"

embeddings_index = {}

with open(glove_path, encoding="utf8") as f:
    for line in f:
        values = line.split()
        word = values[0]
        vector = np.asarray(values[1:], dtype="float32")
        embeddings_index[word] = vector

In [ ]:
vocab_size = len(vocab) + 1

embedding_matrix = np.zeros((vocab_size, embedding_dim))

for word, i in vocab.items():
    vector = embeddings_index.get(word)

    if vector is not None:
        embedding_matrix[i] = vector

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout

model_lstm_glove = Sequential()

model_lstm_glove.add(
    Embedding(
        input_dim=vocab_size,
        output_dim=embedding_dim,
        weights=[embedding_matrix],
        input_length=max_len,
        trainable=True
    )
)

model_lstm_glove.add(LSTM(64))

model_lstm_glove.add(Dropout(0.3))

model_lstm_glove.add(Dense(len(unique_labels), activation="softmax"))

model_lstm_glove.compile(
    loss="sparse_categorical_crossentropy",
    optimizer="adam",
    metrics=["accuracy"]
)

model_lstm_glove.summary()

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True
)

MIN LSTM+GLOVE

In [ ]:
import pandas as pd
import numpy as np

import torch
import torch.nn as nn
import torch.optim as optim

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

from torch.utils.data import Dataset, DataLoader

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [ ]:
train_df = pd.read_csv("/content/trainmerged (4).csv")
val_df = pd.read_csv("/content/valmerged (3).csv")
test_df = pd.read_csv("/content/testmerged (3).csv")

print(train_df.head())

In [ ]:
texts = train_df['Tweet'].astype(str)
labels = train_df['Stance']

tokenizer = Tokenizer(num_words=20000)
tokenizer.fit_on_texts(texts)

X_train = tokenizer.texts_to_sequences(train_df['Tweet'])
X_val = tokenizer.texts_to_sequences(val_df['Tweet'])
X_test = tokenizer.texts_to_sequences(test_df['Tweet'])

max_len = 100

X_train = pad_sequences(X_train, maxlen=max_len)
X_val = pad_sequences(X_val, maxlen=max_len)
X_test = pad_sequences(X_test, maxlen=max_len)

y_train = train_df['Stance'].values
y_val = val_df['Stance'].values
y_test = test_df['Stance'].values

In [ ]:
print(train_df.columns)

In [ ]:
embedding_index = {}

with open("glove.6B.100d.txt", encoding='utf8') as f:
    for line in f:
        values = line.split()
        word = values[0]
        vector = np.asarray(values[1:], dtype='float32')
        embedding_index[word] = vector

In [ ]:
vocab_size = len(tokenizer.word_index) + 1
embedding_dim = 100

embedding_matrix = np.zeros((vocab_size, embedding_dim))

for word, i in tokenizer.word_index.items():

    vector = embedding_index.get(word)

    if vector is not None:
        embedding_matrix[i] = vector

In [ ]:
class TextDataset(Dataset):

    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.long)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

In [ ]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

train_df['Stance'] = le.fit_transform(train_df['Stance'])
val_df['Stance'] = le.transform(val_df['Stance'])
test_df['Stance'] = le.transform(test_df['Stance'])

print(le.classes_)

In [ ]:
y_train = train_df['Stance'].values
y_val = val_df['Stance'].values
y_test = test_df['Stance'].values

In [ ]:
train_dataset = TextDataset(X_train, y_train)
val_dataset = TextDataset(X_val, y_val)
test_dataset = TextDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32)
test_loader = DataLoader(test_dataset, batch_size=32)

In [ ]:
print(type(y_train[0]))
print(y_train[:10])

In [ ]:
class MiniLSTMCell(nn.Module):

    def __init__(self, input_size, hidden_size):
        super(MiniLSTMCell, self).__init__()

        self.hidden_size = hidden_size

        self.Wz = nn.Linear(input_size + hidden_size, hidden_size)
        self.Wh = nn.Linear(input_size + hidden_size, hidden_size)
        self.Wo = nn.Linear(input_size + hidden_size, hidden_size)

    def forward(self, x, h_prev):

        combined = torch.cat((x, h_prev), dim=1)

        z = torch.sigmoid(self.Wz(combined))

        h_tilde = torch.tanh(self.Wh(combined))

        h = (1 - z) * h_prev + z * h_tilde

        o = torch.sigmoid(self.Wo(combined))

        y = o * torch.tanh(h)

        return y

In [ ]:
class MiniLSTMModel(nn.Module):

    def __init__(self, vocab_size, embedding_dim, hidden_size, num_classes, embedding_matrix):

        super(MiniLSTMModel, self).__init__()

        self.embedding = nn.Embedding(vocab_size, embedding_dim)

        self.embedding.weight.data.copy_(torch.from_numpy(embedding_matrix))
        self.embedding.weight.requires_grad = False

        self.minilstm = MiniLSTMCell(embedding_dim, hidden_size)

        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, x):

        embeds = self.embedding(x)

        batch_size = x.size(0)
        h = torch.zeros(batch_size, 128).to(x.device)

        for t in range(embeds.size(1)):
            h = self.minilstm(embeds[:, t, :], h)

        out = self.fc(h)

        return out

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = MiniLSTMModel(
    vocab_size,
    embedding_dim=100,
    hidden_size=128,
    num_classes=3,
    embedding_matrix=embedding_matrix
)

model = model.to(device)

In [ ]:
criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(model.parameters(), lr=0.001)

In [ ]:
import time

start_train = time.time()

epochs = 10
for epoch in range(epochs):
    model.train()
    total_loss = 0

    for X_batch, y_batch in train_loader:

        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()

        outputs = model(X_batch)

        loss = criterion(outputs, y_batch)

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    print("Epoch:", epoch+1, "Loss:", total_loss)

end_train = time.time()

train_time_hours = (end_train - start_train) / 3600
print("Train Time (h):", train_time_hours)

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

model.eval()

preds = []
true = []

with torch.no_grad():

    for X_batch, y_batch in test_loader:

        X_batch = X_batch.to(device)

        outputs = model(X_batch)

        _, predicted = torch.max(outputs, 1)

        preds.extend(predicted.cpu().numpy())
        true.extend(y_batch.numpy())

accuracy = accuracy_score(true, preds)
precision = precision_score(true, preds, average='weighted')
recall = recall_score(true, preds, average='weighted')
f1 = f1_score(true, preds, average='weighted')

print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1:", f1)

In [ ]:
start_inf = time.time()

model.eval()
with torch.no_grad():
    for X_batch, y_batch in test_loader:
        X_batch = X_batch.to(device)
        outputs = model(X_batch)

end_inf = time.time()

infer_time = end_inf - start_inf
print("Inference Time (s):", infer_time)

In [ ]:
total_params = sum(p.numel() for p in model.parameters())
params_million = total_params / 1e6

print("Parameters (M):", params_million)

In [ ]:
results = {
    "Dataset": "VAST",
    "Model": "Mini-LSTM",
    "Accuracy": accuracy,
    "Precision": precision,
    "Recall": recall,
    "F1": f1,
    "Train Time": train_time_hours,
    "Inference Time": infer_time,
    "Params": params_million
}

print(results)

In [ ]:
df = pd.DataFrame([results])
print(df)

In [ ]:
# =========================
# ROC CURVE COMPLETE CODE
# =========================

import torch
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import label_binarize
from sklearn.metrics import roc_curve, auc

# store predictions and true labels
true_labels = []
pred_probs = []

# set model to evaluation mode
model.eval()

with torch.no_grad():

    for X_batch, y_batch in test_loader:

        # move to device (GPU/CPU)
        X_batch = X_batch.to(device)

        # model predictions
        outputs = model(X_batch)

        # convert logits to probabilities
        probabilities = torch.softmax(outputs, dim=1)

        pred_probs.extend(probabilities.cpu().numpy())
        true_labels.extend(y_batch.numpy())

# convert to numpy arrays
true_labels = np.array(true_labels)
pred_probs = np.array(pred_probs)

# number of classes
n_classes = pred_probs.shape[1]

# convert labels to binary (one-vs-rest)
true_labels_bin = label_binarize(true_labels, classes=list(range(n_classes)))

# dictionaries for ROC
fpr = dict()
tpr = dict()
roc_auc = dict()

# compute ROC for each class
for i in range(n_classes):

    fpr[i], tpr[i], _ = roc_curve(true_labels_bin[:, i], pred_probs[:, i])

    roc_auc[i] = auc(fpr[i], tpr[i])

# plot ROC curves
plt.figure(figsize=(7,6))

for i in range(n_classes):

    plt.plot(
        fpr[i],
        tpr[i],
        label=f"Class {i} (AUC = {roc_auc[i]:.3f})"
    )

# random classifier line
plt.plot([0,1],[0,1],'k--')

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve for Stance Detection Model")
plt.legend()
plt.grid()

plt.show()

MIN LSTM+GLOVE+ATTENTION

In [ ]:
class AttentionLayer(nn.Module):

    def __init__(self, hidden_size):
        super(AttentionLayer, self).__init__()

        self.attn = nn.Linear(hidden_size, hidden_size)
        self.v = nn.Linear(hidden_size, 1, bias=False)

    def forward(self, hidden_states):

        energy = torch.tanh(self.attn(hidden_states))

        scores = self.v(energy).squeeze(-1)

        weights = torch.softmax(scores, dim=1)

        context = torch.sum(hidden_states * weights.unsqueeze(-1), dim=1)

        return context

In [ ]:
class MiniLSTM_Attention_Model(nn.Module):

    def __init__(self, vocab_size, embedding_dim, hidden_size, num_classes, embedding_matrix):

        super(MiniLSTM_Attention_Model, self).__init__()

        # GloVe embedding layer
        self.embedding = nn.Embedding(vocab_size, embedding_dim)

        self.embedding.weight.data.copy_(torch.from_numpy(embedding_matrix))

        self.embedding.weight.requires_grad = False

        # MiniLSTM
        self.minilstm = MiniLSTMCell(embedding_dim, hidden_size)

        # Attention
        self.attention = AttentionLayer(hidden_size)

        # Dropout
        self.dropout = nn.Dropout(0.3)

        # Final classifier
        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, x):

        embeds = self.embedding(x)

        batch_size = x.size(0)

        hidden_states = []

        h = torch.zeros(batch_size, 128).to(x.device)

        for t in range(embeds.size(1)):

            h = self.minilstm(embeds[:, t, :], h)

            hidden_states.append(h.unsqueeze(1))

        hidden_states = torch.cat(hidden_states, dim=1)

        context = self.attention(hidden_states)

        context = self.dropout(context)

        out = self.fc(context)

        return out

In [ ]:
model = MiniLSTM_Attention_Model(
    vocab_size,
    embedding_dim=100,
    hidden_size=128,
    num_classes=3,
    embedding_matrix=embedding_matrix
)

model = model.to(device)

In [ ]:
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [ ]:
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',
    factor=0.5,
    patience=2
)

In [ ]:
import time
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# -------------------------------
# TRAINING TIME START
# -------------------------------
train_start = time.time()

epochs = 15
best_val_loss = float('inf')

for epoch in range(epochs):

    model.train()
    total_loss = 0

    for X_batch, y_batch in train_loader:

        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()

        outputs = model(X_batch)

        loss = criterion(outputs, y_batch)

        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

        optimizer.step()

        total_loss += loss.item()

    avg_train_loss = total_loss / len(train_loader)

    # -------- validation ----------
    model.eval()
    val_loss = 0

    with torch.no_grad():

        for X_batch, y_batch in val_loader:

            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            outputs = model(X_batch)

            loss = criterion(outputs, y_batch)

            val_loss += loss.item()

    avg_val_loss = val_loss / len(val_loader)

    scheduler.step(avg_val_loss)

    print(f"Epoch {epoch+1}")
    print("Train Loss:", avg_train_loss)
    print("Val Loss:", avg_val_loss)

# -------------------------------
# TRAINING TIME END
# -------------------------------
train_end = time.time()

train_time_hours = (train_end - train_start) / 3600

print("Training Time (hours):", train_time_hours)


# -------------------------------
# INFERENCE TIME
# -------------------------------
infer_start = time.time()

model.eval()

preds = []
true = []

with torch.no_grad():

    for X_batch, y_batch in test_loader:

        X_batch = X_batch.to(device)

        outputs = model(X_batch)

        _, predicted = torch.max(outputs, 1)

        preds.extend(predicted.cpu().numpy())
        true.extend(y_batch.numpy())

infer_end = time.time()

infer_time_seconds = infer_end - infer_start

print("Inference Time (seconds):", infer_time_seconds)


# -------------------------------
# METRICS
# -------------------------------
accuracy = accuracy_score(true, preds)

precision = precision_score(true, preds, average='weighted')

recall = recall_score(true, preds, average='weighted')

f1 = f1_score(true, preds, average='weighted')

print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1 Score:", f1)


# -------------------------------
# PARAMETER COUNT
# -------------------------------
total_params = sum(p.numel() for p in model.parameters())

params_million = total_params / 1e6

print("Parameters (M):", params_million)

In [ ]:
import pandas as pd

results = {
    "Dataset": "VAST",
    "Model": "Mini-LSTM + Attention",
    "Accuracy": accuracy,
    "Precision": precision,
    "Recall": recall,
    "F1-Score": f1,
    "Train Time (h)": train_time_hours,
    "Infer (s)": infer_time_seconds,
    "Params (M)": params_million
}

# print dictionary
print(results)

# convert to dataframe
df = pd.DataFrame([results])

# display table
print(df)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.metrics import roc_curve, auc
from sklearn.preprocessing import label_binarize
import torch

model.eval()

true_labels = []
pred_probs = []

with torch.no_grad():

    for X_batch, y_batch in test_loader:

        X_batch = X_batch.to(device)

        outputs = model(X_batch)

        probabilities = torch.softmax(outputs, dim=1)

        pred_probs.extend(probabilities.cpu().numpy())
        true_labels.extend(y_batch.numpy())

true_labels = np.array(true_labels)
pred_probs = np.array(pred_probs)

# number of classes
n_classes = pred_probs.shape[1]

# convert labels to binary
true_labels_bin = label_binarize(true_labels, classes=list(range(n_classes)))

fpr = {}
tpr = {}
roc_auc = {}

for i in range(n_classes):

    fpr[i], tpr[i], _ = roc_curve(true_labels_bin[:, i], pred_probs[:, i])
    roc_auc[i] = auc(fpr[i], tpr[i])

# Plot ROC
plt.figure(figsize=(7,6))

for i in range(n_classes):
    plt.plot(fpr[i], tpr[i], label=f"Class {i} (AUC = {roc_auc[i]:.3f})")

plt.plot([0,1],[0,1],'k--')

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend()
plt.grid()

plt.show()

# Average AUC
avg_auc = np.mean(list(roc_auc.values()))

print("Average AUC:", avg_auc)

# Store in results dictionary
roc_results = {
    "Dataset": "VAST",
    "Model": "Mini-LSTM + Attention",
    "ROC-AUC": avg_auc
}

print(roc_results)

df_roc = pd.DataFrame([roc_results])
print(df_roc)

MIN GRU+GLOVE EMBEDDINGS

In [ ]:
import torch
import torch.nn as nn

class MiniGRUCell(nn.Module):

    def __init__(self, input_size, hidden_size):
        super(MiniGRUCell, self).__init__()

        self.hidden_size = hidden_size

        # update gate
        self.Wz = nn.Linear(input_size + hidden_size, hidden_size)

        # candidate hidden state
        self.Wh = nn.Linear(input_size + hidden_size, hidden_size)

    def forward(self, x, h_prev):

        combined = torch.cat((x, h_prev), dim=1)

        z = torch.sigmoid(self.Wz(combined))

        h_tilde = torch.tanh(self.Wh(combined))

        h = (1 - z) * h_prev + z * h_tilde

        return h

In [ ]:
class MiniGRUModel(nn.Module):

    def __init__(self, vocab_size, embedding_dim, hidden_size, num_classes, embedding_matrix):

        super(MiniGRUModel, self).__init__()

        # GloVe embedding layer
        self.embedding = nn.Embedding(vocab_size, embedding_dim)

        self.embedding.weight.data.copy_(torch.from_numpy(embedding_matrix))

        self.embedding.weight.requires_grad = False

        # MiniGRU
        self.minigru = MiniGRUCell(embedding_dim, hidden_size)

        # dropout
        self.dropout = nn.Dropout(0.3)

        # classifier
        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, x):

        embeds = self.embedding(x)

        batch_size = x.size(0)

        h = torch.zeros(batch_size, 128).to(x.device)

        for t in range(embeds.size(1)):

            h = self.minigru(embeds[:, t, :], h)

        h = self.dropout(h)

        out = self.fc(h)

        return out

In [ ]:
model = MiniGRUModel(
    vocab_size,
    embedding_dim=100,
    hidden_size=128,
    num_classes=3,
    embedding_matrix=embedding_matrix
)

model = model.to(device)

In [ ]:
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [ ]:
import time
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# -----------------------------
# TRAINING START TIME
# -----------------------------
train_start = time.time()

epochs = 15

for epoch in range(epochs):

    model.train()
    total_loss = 0

    for X_batch, y_batch in train_loader:

        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()

        outputs = model(X_batch)

        loss = criterion(outputs, y_batch)

        loss.backward()

        # gradient clipping for RNN stability
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

        optimizer.step()

        total_loss += loss.item()

    avg_train_loss = total_loss / len(train_loader)

    # -----------------------------
    # VALIDATION
    # -----------------------------
    model.eval()
    val_loss = 0

    with torch.no_grad():

        for X_batch, y_batch in val_loader:

            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            outputs = model(X_batch)

            loss = criterion(outputs, y_batch)

            val_loss += loss.item()

    avg_val_loss = val_loss / len(val_loader)

    print(f"Epoch {epoch+1}")
    print("Train Loss:", avg_train_loss)
    print("Val Loss:", avg_val_loss)


# -----------------------------
# TRAINING TIME
# -----------------------------
train_end = time.time()

train_time_hours = (train_end - train_start) / 3600

print("Training Time (hours):", train_time_hours)


# -----------------------------
# INFERENCE TIME + PREDICTIONS
# -----------------------------
infer_start = time.time()

model.eval()

preds = []
true = []

with torch.no_grad():

    for X_batch, y_batch in test_loader:

        X_batch = X_batch.to(device)

        outputs = model(X_batch)

        _, predicted = torch.max(outputs, 1)

        preds.extend(predicted.cpu().numpy())
        true.extend(y_batch.numpy())

infer_end = time.time()

infer_time_seconds = infer_end - infer_start

print("Inference Time (seconds):", infer_time_seconds)


# -----------------------------
# METRICS
# -----------------------------
accuracy = accuracy_score(true, preds)

precision = precision_score(true, preds, average='weighted')

recall = recall_score(true, preds, average='weighted')

f1 = f1_score(true, preds, average='weighted')

print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1 Score:", f1)


# -----------------------------
# PARAMETER COUNT
# -----------------------------
total_params = sum(p.numel() for p in model.parameters())

params_million = total_params / 1e6

print("Parameters (M):", params_million)

In [ ]:
import pandas as pd

results = {
    "Dataset": "VAST",
    "Model": "Mini-GRU",
    "Accuracy": accuracy,
    "Precision": precision,
    "Recall": recall,
    "F1": f1,
    "Train Time": train_time_hours,
    "Inference Time": infer_time_seconds,
    "Params": params_million
}

df = pd.DataFrame([results])
print(df)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import label_binarize
from sklearn.metrics import roc_curve, auc
import torch

model.eval()

true_labels = []
pred_probs = []

# Get predictions
with torch.no_grad():

    for X_batch, y_batch in test_loader:

        X_batch = X_batch.to(device)

        outputs = model(X_batch)

        probabilities = torch.softmax(outputs, dim=1)

        pred_probs.extend(probabilities.cpu().numpy())
        true_labels.extend(y_batch.numpy())

true_labels = np.array(true_labels)
pred_probs = np.array(pred_probs)

# Number of classes
n_classes = pred_probs.shape[1]

# Convert labels to binary
true_labels_bin = label_binarize(true_labels, classes=list(range(n_classes)))

fpr = {}
tpr = {}
roc_auc = {}

# Compute ROC
for i in range(n_classes):

    fpr[i], tpr[i], _ = roc_curve(true_labels_bin[:, i], pred_probs[:, i])
    roc_auc[i] = auc(fpr[i], tpr[i])

# Plot ROC curves
plt.figure(figsize=(7,6))

for i in range(n_classes):

    plt.plot(
        fpr[i],
        tpr[i],
        label=f"Class {i} (AUC = {roc_auc[i]:.3f})"
    )

# random classifier line
plt.plot([0,1],[0,1],'k--')

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve - MiniGRU Model")
plt.legend()
plt.grid()

plt.show()

# Average AUC
avg_auc = np.mean(list(roc_auc.values()))

print("Average ROC AUC:", avg_auc)

In [ ]:
import torch
import torch.nn as nn

class AttentionLayer(nn.Module):

    def __init__(self, hidden_size):
        super(AttentionLayer, self).__init__()

        self.attn = nn.Linear(hidden_size, hidden_size)
        self.v = nn.Linear(hidden_size, 1, bias=False)

    def forward(self, hidden_states):

        energy = torch.tanh(self.attn(hidden_states))

        scores = self.v(energy).squeeze(-1)

        weights = torch.softmax(scores, dim=1)

        context = torch.sum(hidden_states * weights.unsqueeze(-1), dim=1)

        return context

In [ ]:
class MiniGRUCell(nn.Module):

    def __init__(self, input_size, hidden_size):
        super(MiniGRUCell, self).__init__()

        self.Wz = nn.Linear(input_size + hidden_size, hidden_size)
        self.Wh = nn.Linear(input_size + hidden_size, hidden_size)

    def forward(self, x, h_prev):

        combined = torch.cat((x, h_prev), dim=1)

        z = torch.sigmoid(self.Wz(combined))

        h_tilde = torch.tanh(self.Wh(combined))

        h = (1 - z) * h_prev + z * h_tilde

        return h

In [ ]:
class MiniGRU_Attention_Model(nn.Module):

    def __init__(self, vocab_size, embedding_dim, hidden_size, num_classes, embedding_matrix):

        super(MiniGRU_Attention_Model, self).__init__()

        # GloVe embedding
        self.embedding = nn.Embedding(vocab_size, embedding_dim)

        self.embedding.weight.data.copy_(torch.from_numpy(embedding_matrix))

        self.embedding.weight.requires_grad = False

        # MiniGRU
        self.minigru = MiniGRUCell(embedding_dim, hidden_size)

        # Attention
        self.attention = AttentionLayer(hidden_size)

        # Dropout
        self.dropout = nn.Dropout(0.3)

        # Output layer
        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, x):

        embeds = self.embedding(x)

        batch_size = x.size(0)

        hidden_states = []

        h = torch.zeros(batch_size, 128).to(x.device)

        for t in range(embeds.size(1)):

            h = self.minigru(embeds[:, t, :], h)

            hidden_states.append(h.unsqueeze(1))

        hidden_states = torch.cat(hidden_states, dim=1)

        context = self.attention(hidden_states)

        context = self.dropout(context)

        out = self.fc(context)

        return out

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

vocab_size = len(tokenizer.word_index) + 1

model = MiniGRU_Attention_Model(
    vocab_size,
    embedding_dim=100,
    hidden_size=128,
    num_classes=3,
    embedding_matrix=embedding_matrix
)

model = model.to(device)

In [ ]:
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [ ]:
import time

train_start = time.time()

epochs = 15

for epoch in range(epochs):

    model.train()
    total_loss = 0

    for X_batch, y_batch in train_loader:

        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()

        outputs = model(X_batch)

        loss = criterion(outputs, y_batch)

        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

        optimizer.step()

        total_loss += loss.item()

    avg_train_loss = total_loss / len(train_loader)

    model.eval()
    val_loss = 0

    with torch.no_grad():

        for X_batch, y_batch in val_loader:

            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            outputs = model(X_batch)

            loss = criterion(outputs, y_batch)

            val_loss += loss.item()

    avg_val_loss = val_loss / len(val_loader)

    print("Epoch:", epoch+1)
    print("Train Loss:", avg_train_loss)
    print("Val Loss:", avg_val_loss)

train_end = time.time()

train_time_hours = (train_end - train_start) / 3600

print("Training Time (hours):", train_time_hours)

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

infer_start = time.time()

model.eval()

preds = []
true = []

with torch.no_grad():

    for X_batch, y_batch in test_loader:

        X_batch = X_batch.to(device)

        outputs = model(X_batch)

        _, predicted = torch.max(outputs, 1)

        preds.extend(predicted.cpu().numpy())
        true.extend(y_batch.numpy())

infer_end = time.time()

infer_time_seconds = infer_end - infer_start

accuracy = accuracy_score(true, preds)
precision = precision_score(true, preds, average='weighted')
recall = recall_score(true, preds, average='weighted')
f1 = f1_score(true, preds, average='weighted')

print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1 Score:", f1)
print("Inference Time:", infer_time_seconds)

In [ ]:
total_params = sum(p.numel() for p in model.parameters())

params_million = total_params / 1e6

print("Parameters (M):", params_million)

In [ ]:
import pandas as pd

results = {
"Dataset":"VAST",
"Model":"MiniGRU + Attention",
"Accuracy":accuracy,
"Precision":precision,
"Recall":recall,
"F1":f1,
"Train Time":train_time_hours,
"Inference Time":infer_time_seconds,
"Params":params_million
}

df = pd.DataFrame([results])

print(df)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import label_binarize
from sklearn.metrics import roc_curve, auc

model.eval()

true_labels = []
pred_probs = []

with torch.no_grad():

    for X_batch, y_batch in test_loader:

        X_batch = X_batch.to(device)

        outputs = model(X_batch)

        probabilities = torch.softmax(outputs, dim=1)

        pred_probs.extend(probabilities.cpu().numpy())
        true_labels.extend(y_batch.numpy())

true_labels = np.array(true_labels)
pred_probs = np.array(pred_probs)

# number of classes
n_classes = pred_probs.shape[1]

# convert labels to binary
true_labels_bin = label_binarize(true_labels, classes=list(range(n_classes)))

fpr = {}
tpr = {}
roc_auc = {}

for i in range(n_classes):

    fpr[i], tpr[i], _ = roc_curve(true_labels_bin[:, i], pred_probs[:, i])
    roc_auc[i] = auc(fpr[i], tpr[i])

# plot ROC
plt.figure(figsize=(7,6))

for i in range(n_classes):

    plt.plot(
        fpr[i],
        tpr[i],
        label=f"Class {i} (AUC = {roc_auc[i]:.3f})"
    )

plt.plot([0,1],[0,1],'k--')

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve - MiniGRU + Attention")
plt.legend()
plt.grid()

plt.show()

In [ ]:
!pip install transformers

In [ ]:
import torch
import torch.nn as nn
import time
import pandas as pd

from transformers import BertTokenizer, BertModel
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

In [ ]:
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

In [ ]:
class BertDataset(torch.utils.data.Dataset):

    def __init__(self, texts, labels, tokenizer, max_len=128):

        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):

        text = str(self.texts[idx])
        label = self.labels[idx]

        encoding = self.tokenizer(
            text,
            padding='max_length',
            truncation=True,
            max_length=self.max_len,
            return_tensors='pt'
        )

        input_ids = encoding['input_ids'].squeeze(0)
        attention_mask = encoding['attention_mask'].squeeze(0)

        return input_ids, attention_mask, torch.tensor(label)

In [ ]:
train_dataset = BertDataset(train_df['Tweet'].values, y_train, tokenizer)
val_dataset = BertDataset(val_df['Tweet'].values, y_val, tokenizer)
test_dataset = BertDataset(test_df['Tweet'].values, y_test, tokenizer)

In [ ]:
from torch.utils.data import DataLoader

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16)
test_loader = DataLoader(test_dataset, batch_size=16)

In [ ]:
class BertClassifier(nn.Module):

    def __init__(self, num_classes):

        super(BertClassifier, self).__init__()

        self.bert = BertModel.from_pretrained('bert-base-uncased')

        self.dropout = nn.Dropout(0.3)

        self.fc = nn.Linear(768, num_classes)

    def forward(self, input_ids, attention_mask):

        outputs = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        cls_output = outputs.pooler_output

        x = self.dropout(cls_output)

        out = self.fc(x)

        return out

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = BertClassifier(num_classes=3)

model = model.to(device)

In [ ]:
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(model.parameters(), lr=2e-5)

In [ ]:
train_start = time.time()

epochs = 3

for epoch in range(epochs):

    model.train()

    total_loss = 0

    for input_ids, attention_mask, labels in train_loader:

        input_ids = input_ids.to(device)
        attention_mask = attention_mask.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(input_ids, attention_mask)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    print("Epoch:", epoch+1, "Loss:", total_loss)

train_end = time.time()

train_time_hours = (train_end - train_start)/3600

print("Training Time:", train_time_hours)